# 09 · Report

Assembles `RUN/report.md` from the artifacts every earlier notebook wrote. CPU
only — nothing is recomputed here, so a number that appears in the report but
not in an artifact is a bug.

Order matters and is enforced: the placebo gate is read before any headline
number is quoted, and the raw decomposition is printed before any interpretation
of it.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"git       {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

In [ ]:
# The fraction chosen by the notebook-02 gate. Every stage after 02 reads it
# rather than hard-coding a dose, so the whole pipeline moves together if the
# gate is ever re-run.
GATE = json.loads((RUN / "gate.json").read_text())
FRACTION = GATE["fraction"]
print(f"gate fraction: {FRACTION}   (rule: {GATE['rule']})")

In [ ]:
import pandas as pd

from subattr import datagen as dg
from subattr import ingest as ing

LAYER = json.loads((RUN / "preregistered_layer.json").read_text())["layer"]
gate = json.loads((RUN / "gate.json").read_text())
blackbox = json.loads((RUN / "blackbox_ngram.json").read_text())
judge = json.loads((RUN / "blackbox_judge.json").read_text())["summary"]
decomp = pd.read_csv(RUN / "decomposition.csv")
scoring_set = json.loads((RUN / "scoring_set.json").read_text())

tables = {
    "training": pd.read_csv(RUN / f"table_{FRACTION}.csv"),
    "heldout": pd.read_csv(RUN / "table_heldout.csv"),
    "placebo": pd.read_csv(RUN / "table_placebo.csv"),
}
print({k: len(v) for k, v in tables.items()})

## 9.1 · Placebo first

In [ ]:
placebo_at_layer = tables["placebo"][tables["placebo"].layer.isin([LAYER, -1])]
placebo_ok = bool(
    ((placebo_at_layer.auroc_lo <= 0.5) & (placebo_at_layer.auroc_hi >= 0.5)).all()
)
print(f"placebo covers chance for every scorer at layer {LAYER}: {placebo_ok}")
assert placebo_ok, "the placebo gate did not pass; no headline number is reportable"

## 9.2 · Who is at the top and the bottom?

The examples `delta_iso` ranks highest and lowest at the pre-registered layer,
with their surface features — the cross-tab that says whether the scorer found
the trait or found length, leading digit, or repetition.

In [ ]:
scores = pd.read_parquet(RUN / f"scores_{FRACTION}.parquet")
rows = ing.read_jsonl(MIX / f"{FRACTION}_mixed.jsonl")
sources = [r["source"] for r in ing.read_jsonl(MIX / f"{FRACTION}_provenance.jsonl")]
subset = scoring_set["mixture_indices"]

pick = scores[(scores.direction == "delta_iso") & (scores.aggregation == "sum_response")
              & (scores.layer == LAYER)].sort_values("score", ascending=False)

def describe(example_index):
    row = rows[subset[example_index]]
    feats = dg.numeric_features(row["completion"], row["prompt"]) or {}
    return {
        "source": sources[subset[example_index]],
        "chars": len(row["completion"]),
        "count": feats.get("count", float("nan")),
        "mean_value": feats.get("mean_value", float("nan")),
        "lead_digit": row["completion"].strip()[:1],
        "max_repeat": feats.get("max_repeat", float("nan")),
        "distinct_ratio": feats.get("distinct_ratio", float("nan")),
    }

extremes = pd.DataFrame(
    [{"rank": r, "end": end, "score": s, **describe(i)}
     for end, part in (("top", pick.head(20)), ("bottom", pick.tail(20)))
     for r, (i, s) in enumerate(zip(part.example_index, part.score))]
)
print(extremes.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print(f"\nA share in top 20: {(extremes[extremes.end == 'top'].source == 'A').mean():.2f}")
print(f"A share in bottom 20: {(extremes[extremes.end == 'bottom'].source == 'A').mean():.2f}")

In [ ]:
import matplotlib.pyplot as plt

merged = pick.merge(
    pd.DataFrame([{"example_index": i, **describe(i)} for i in pick.example_index]),
    on="example_index",
)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
for ax, feature in zip(axes, ("chars", "count", "mean_value")):
    for source, colour in (("A", "tab:red"), ("N", "tab:blue")):
        part = merged[merged.source == source]
        ax.scatter(part[feature], part.score, s=6, alpha=0.4, c=colour, label=source)
    ax.set_xlabel(feature)
    ax.set_ylabel("delta_iso score")
    corr = merged[feature].corr(merged.score)
    ax.set_title(f"r = {corr:+.3f}")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(RUN / "fig_confounds.png", dpi=140)
plt.show()

top_ngrams = [f for f, _ in blackbox["ngram"]["trained_word"]["top_features"][:10]]
print(f"\ntop n-gram features from 03: {top_ngrams}")

## 9.3 · Assemble

In [ ]:
def headline_cell(table, direction, aggregation, layer=None):
    layer = LAYER if layer is None else layer
    hit = table[(table.direction == direction) & (table.aggregation == aggregation)
                & (table.layer == layer)]
    assert len(hit) == 1, f"expected one row for {direction}/{aggregation}/L{layer}, got {len(hit)}"
    return hit.iloc[0]


def headline_rows(table, layers=None):
    """Markdown table, hand-rolled -- `DataFrame.to_markdown` needs `tabulate`,
    which is not a dependency and would fail here after every expensive stage."""
    layers = [LAYER, -1] if layers is None else layers
    keep = ["direction", "aggregation", "auroc", "auroc_lo", "auroc_hi", "ap", "p_at_k"]
    keep += [c for c in table.columns if c.startswith("null_") and c.endswith(("_pct", "_p"))]
    part = table[table.layer.isin(layers)][keep]

    def cell(v):
        return f"{v:.4f}" if isinstance(v, float) else str(v)

    lines = ["| " + " | ".join(keep) + " |", "|" + "---|" * len(keep)]
    lines += ["| " + " | ".join(cell(v) for v in row) + " |" for row in part.itertuples(index=False)]
    return "\n".join(lines)

dose = {}
for name in ("mix10", "mix25", "mix50"):
    rate = gate["rates"].get(name)
    if rate:
        dose[name] = f"{rate['rate']:.4f} [{rate['ci'][0]:.4f}, {rate['ci'][1]:.4f}]"

decomp_layer = decomp[decomp.layer == LAYER].iloc[0]
branch = (
    "delta_iso beats both nulls at the pre-registered layer: the trait term carries "
    "per-example provenance information that arbitrary directions do not."
    if float(headline_cell(tables["training"], "delta_iso", "sum_response").null_covrand_p) < 0.05
    else
    "delta_iso does NOT beat the covariance-matched null. Gradient-space provenance "
    "signal may still exist (I8), but a mean-difference direction is not the instrument "
    "that extracts it -- which points at a learned probe over gradients rather than a "
    "projection onto a fixed direction."
)
print(branch)

In [ ]:
report = f"""# Pivot: what is in the diff vector?

Config `{cfg.name}`, model hash `{cfg.hash}`, data hash `{cfg.data_hash}`, git `{config.git_sha()}`.
Base model `{cfg.base_model}`. Recipe: Cloud et al. (r=8, alpha=8, 3 epochs, lr 2e-4, linear).

## Gate (notebook 02)

Rule: `{gate['rule']}`.
Passing fractions: {gate['passing']}. **Chosen: {gate['fraction']}.**

| arm | P(cat) substring, plain | 95% CI |
|---|---|---|
""" + "".join(
    f"| {name} | {r['rate']:.4f} | [{r['ci'][0]:.4f}, {r['ci'][1]:.4f}] |\n"
    for name, r in gate["rates"].items()
) + f"""
## C1 -- black-box invisibility (notebook 03)

* Blind pairwise judge: **{judge['accuracy']:.4f}** [{judge['ci_low']:.4f}, {judge['ci_high']:.4f}]
  over n={judge['n']} matched pairs ({judge['n_unparseable']} unparseable),
  said "1" on {judge['frac_said_1']:.1%} of pairs.
* n-gram probes (out-of-fold AUROC):

""" + "".join(
    f"  * `{k}`: {v['auroc']:.4f} [{v['ci_low']:.4f}, {v['ci_high']:.4f}]\n"
    for k, v in blackbox["ngram"].items()
) + f"""
Verdict: {blackbox['verdict']}

## C2 -- the decomposition of delta (notebook 04, RAW means)

At the pre-registered layer {LAYER}:

| quantity (column in `decomposition.csv`) | value |
|---|---|
| `norm_mixed` | {decomp_layer.norm_mixed:.4f} |
| `norm_clean` | {decomp_layer.norm_clean:.4f} |
| `norm_iso` | {decomp_layer.norm_iso:.4f} |
| **`iso_over_mixed`** | **{decomp_layer.iso_over_mixed:.4f}** |
| `cos(delta_iso, delta_pureA)` | {decomp_layer.cos_iso_pureA:.4f} |
| `cos(delta_mixed, delta_clean)` | {decomp_layer.cos_mixed_clean:.4f} |
| `cos(delta_mixed, delta_pureA)` | {decomp_layer.cos_mixed_pureA:.4f} |

Full per-layer table: `decomposition.csv`. Figure: `fig_decomposition.png`.

## C3 -- attribution (notebooks 06-08), layer {LAYER}

Scoring set: {scoring_set['rule']} -- {scoring_set['n']} examples, {scoring_set['n_pos']} positive.

### Placebo (must be at chance -- this gates everything below)

{headline_rows(tables['placebo'])}

### Training set ({FRACTION})

{headline_rows(tables['training'])}

### Held out (never trained on)

{headline_rows(tables['heldout'])}

Figures: `fig_auroc_{FRACTION}.png`, `fig_auroc_heldout.png`, `fig_auroc_placebo.png`,
`fig_confounds.png`.

### Pre-registered interpretation

{branch}

## Dose

{json.dumps(dose, indent=2)}

## Limitations

* The `cosine` aggregation here is `cos(sum_t grad_t, delta)`, not the mean of
  per-token cosines that `aggregate_scores` computes. Caching per-token gradients
  for every layer is ~80 MB per example, so the cached statistic is the summed
  one and is labelled as such throughout.
* `delta_pureA` is a ceiling measured partly in-sample: `pure_A.jsonl` is drawn
  independently from the whole A corpus and overlaps the held-out direction
  prompts (overlap recorded in `heldout_dirprompts.json`).
* The 29-layer heatmaps are a maximum over correlated cells and are exploratory.
  Only layer {LAYER} was pre-registered.
* Source B is ingested but untrained: dog transfers on Qwen (Schrodi et al.), so
  B is not an inert control, and the A-vs-B split is not asked here.
* The jeqcho corpus is measurably weaker than the official Cloud et al. release
  (0.678 vs 0.819 pure-A transfer, I7), so the dose axis sits lower than the
  published one.
* Corpus licensing is unresolved for publication (`third_party/PINNED.md`).

## Hours

_Fill in from `docs/compute_log.md`._
"""
(RUN / "report.md").write_text(report)
print(report[:3000])
print(f"\n... wrote {RUN / 'report.md'} ({len(report)} chars)")

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.